Datasetlink: https://www.kaggle.com/datasets/sumedh1507/teen-phone-addiction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.read_csv("teen_phone_addiction_dataset.csv")

In [ ]:
df.drop(columns=[
    'ID', 'Name', 'Location', 'School_Grade', 'Academic_Performance'
], inplace=True)

In [ ]:
df['Addiction_Level'] = pd.cut(
    df['Addiction_Level'],
    bins=[0, 4, 7, 10],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
target_encoder = LabelEncoder()
df['Addiction_Level'] = target_encoder.fit_transform(df['Addiction_Level'])

In [ ]:
label_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
X = df.drop("Addiction_Level", axis=1)
y = df["Addiction_Level"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
base_models = [
    ("dt", DecisionTreeClassifier(criterion='entropy', random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=500, max_depth=None, bootstrap=False, random_state=42)),
    ("gb", GradientBoostingClassifier(n_estimators=500, max_depth=20, random_state=42))
]

meta_model = LogisticRegression(C=1e5)

In [ ]:
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv='prefit' if hasattr(StackingClassifier, 'prefit') else 2, 
    passthrough=True 
)

In [ ]:
stack_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = stack_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print("\n==============================")
print("FINAL MODEL PERFORMANCE")
print("==============================")
print(f"\nAccuracy: {accuracy * 100:.2f}%")

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

C:\Users\DELL\AppData\Local\Temp\ipykernel_15068\691228835.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns



FINAL MODEL PERFORMANCE

Accuracy: 99.00%

Classification Report

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       511
           1       1.00      0.75      0.86         8
           2       0.94      0.99      0.96        81

    accuracy                           0.99       600
   macro avg       0.98      0.91      0.94       600
weighted avg       0.99      0.99      0.99       600


Confusion Matrix

[[508   0   3]
 [  0   6   2]
 [  1   0  80]]
